In [1]:
from collections import Counter
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


data = pd.read_excel(r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage3\397×87.xlsx")
X = data.iloc[:, 3:]
X
feature_names = X.columns
X = StandardScaler().fit_transform(X)

min_val = np.min(X, axis=0)
max_val = np.max(X, axis=0)
X = (X - min_val) / (max_val - min_val)
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)
X.shape

(397, 87)

In [2]:

def extract_numbers(file_path):
    df = pd.read_excel(file_path)
    valid_numbers = []
    for index, value in df['composition'].items():
        if not pd.isna(value):
            valid_numbers.append(int(df.at[index, 'number']))
    return valid_numbers

file_path = r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage3\benchmark3.xlsx"
reported_structures = extract_numbers(file_path)

for j in range(10, 100, 5):
    j = j/100

    pca = PCA(n_components= j)
    pca_data = pca.fit_transform(X)
    num_pca = pca_data.shape[1]
    for i in range(3, 7):
        #for n in ['euclidean', 'cosine', 'manhattan', 'precomputed']:
        #    for k in ['ward', 'complete', 'average', 'single']:
                model = AgglomerativeClustering(n_clusters= i, metric = 'cosine', linkage = 'single')
                model = model.fit(pca_data)
                labels = model.labels_
                if len(np.unique(labels)) > 1:
                    #silhouette_avg = silhouette_score(new_data, labels)
                    #calinski_harabasz_avg = calinski_harabasz_score(new_data, labels)
                    print(f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")
                else:
                    pass

                data[f'cluster_{i}'] = labels
                print(f"Number of clusters k={i}:",f"\nPCA={j}:")
                dict = {}

                label_1 = []
                for num  in  reported_structures:
                    label = labels[num-1]
                    label_1.append(int(label))
                print(label_1)

                for cluster in range(i):

                    cluster_samples = data[data[f'cluster_{i}'] == cluster]
                        # print(cluster_samples)

                    total_in_cluster = len(cluster_samples)


                    reported_in_cluster = len([num for num in cluster_samples['number'] if num in reported_structures])
                    dict[cluster] = (reported_in_cluster, total_in_cluster)
                for value in dict.values():
                    if value[0] == 0:
                        for key, value in dict.items():

                            formatted_value = f"{value[0]}/{value[1]}"

                            print(f"Cluster {key}: {formatted_value}")
                            #print(f"Cluster {cluster}: {reported_in_cluster}/{total_in_cluster} "
                            #    f"(Reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


Silhouette score: 0.28 
Calinski-Harabasz score: 1505.88
Number of clusters k=3: 
PCA=0.1:
[0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0]
Cluster 0: 8/182
Cluster 1: 7/214
Cluster 2: 0/1

Silhouette score: 0.27 
Calinski-Harabasz score: 1001.39
Number of clusters k=4: 
PCA=0.1:
[0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0]
Cluster 0: 8/181
Cluster 1: 7/214
Cluster 2: 0/1
Cluster 3: 0/1
Cluster 0: 8/181
Cluster 1: 7/214
Cluster 2: 0/1
Cluster 3: 0/1

Silhouette score: 0.26 
Calinski-Harabasz score: 749.13
Number of clusters k=5: 
PCA=0.1:
[0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0]
Cluster 0: 8/180
Cluster 1: 7/214
Cluster 2: 0/1
Cluster 3: 0/1
Cluster 4: 0/1
Cluster 0: 8/180
Cluster 1: 7/214
Cluster 2: 0/1
Cluster 3: 0/1
Cluster 4: 0/1
Cluster 0: 8/180
Cluster 1: 7/214
Cluster 2: 0/1
Cluster 3: 0/1
Cluster 4: 0/1

Silhouette score: 0.25 
Calinski-Harabasz score: 597.82
Number of clusters k=6: 
PCA=0.1:
[0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0]
Cluster 0: 8/179
Cluster 1: 7/214
Cl

In [3]:
pca = PCA(n_components= 0.85, random_state=42)
new_data = pca.fit_transform(X)
print(new_data.shape)
ratios = pca.explained_variance_ratio_
components = pca.components_
X = pd.DataFrame(X)

components_df = pd.DataFrame(
    components,
    columns=X.columns,
    index=[f"PC{i+1}" for i in range(components.shape[0])]
)

output_file = "pca_components.xlsx"
components_df.to_excel(output_file, index=True)


i = 6
model = AgglomerativeClustering(n_clusters= i, metric = 'manhattan', linkage = 'complete')
model = model.fit(new_data)
labels = model.labels_

df = pd.DataFrame(labels)
df.to_excel("ward_result_1.xlsx", index=False)

data[f'cluster_{i}'] = labels

print(f"\nNumber of clusters k={i}:")
print(f"\nSilhouette score: {silhouette_score(new_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(new_data, labels):.2f}")
label_1 = []
for num  in  reported_structures:
        label = labels[num-1]
        label_1.append(int(label))
print(label_1)


for cluster in range(i):

    cluster_samples = data[data[f'cluster_{i}'] == cluster]
            # print(cluster_samples)

    total_in_cluster = len(cluster_samples)
    print(total_in_cluster)

    reported_in_cluster = len([num for num in cluster_samples['number'] if num in reported_structures])

    print(f"Cluster {cluster}: {reported_in_cluster}/{total_in_cluster} "
                f"(Reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


(397, 21)

Number of clusters k=6:

Silhouette score: 0.11 
Calinski-Harabasz score: 44.01
[0, 0, 1, 3, 1, 3, 3, 3, 3, 1, 1, 3, 0, 0, 0]
120
Cluster 0: 5/120 (Reported ratio: 4.17%)
64
Cluster 1: 4/64 (Reported ratio: 6.25%)
28
Cluster 2: 0/28 (Reported ratio: 0.00%)
165
Cluster 3: 6/165 (Reported ratio: 3.64%)
14
Cluster 4: 0/14 (Reported ratio: 0.00%)
6
Cluster 5: 0/6 (Reported ratio: 0.00%)


In [4]:
import pandas as pd
from sklearn.decomposition import PCA
from openpyxl.styles import Font
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import Workbook


wb = Workbook()
ws = wb.active


for r in dataframe_to_rows(components_df, index=True, header=True):
    ws.append(r)


red_bold_font = Font(color="FF0000", bold=True)


for row in range(2, ws.max_row + 1):

    values = [ws.cell(row=row, column=col).value for col in range(2, ws.max_column + 1)]


    values = [0 if v is None else v for v in values]


    abs_values = [abs(v) for v in values]
    top_10_indices = sorted(range(len(abs_values)), key=lambda i: abs_values[i], reverse=True)[:10]


    for idx in top_10_indices:
        ws.cell(row=row, column=idx + 2).font = red_bold_font


output_file = "pca_components_highlighted.xlsx"
wb.save(output_file)

print(f"The component-loading matrix has been saved to {output_file}, with the 10 largest absolute values highlighted.")

The component-loading matrix has been saved to pca_components_highlighted.xlsx, with the 10 largest absolute values highlighted.
